# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library, following best practices for reproducible and programmatic data science.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if not already installed in your environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")
print(f"Version: {getattr(meta, 'version', None)}")

## 2. Data Overview
Examine available record sets, their fields, and IDs. All entities are referenced strictly by their `@id` fields.

In [ ]:
record_sets = []
try:
    # Not all schemas use recordSet with direct list; fallback for .record_sets property
    record_sets = [rs['@id'] for rs in getattr(meta, 'recordSet', [])]
    if not record_sets and hasattr(meta, 'record_sets'):
        record_sets = [rs['@id'] for rs in meta.record_sets]
except Exception as exc:
    pass
# If record_sets is still empty, try via dataset.record_sets (mlcroissant API)
if not record_sets:
    try:
        record_sets = [rs['@id'] for rs in dataset.record_sets()]
    except Exception:
        # As last resort, look for record sets by introspecting object
        if hasattr(meta, 'record_sets'):
            record_sets = [rs['@id'] for rs in meta.record_sets]

print("Record Sets (by @id):")
for rs_id in record_sets:
    print(f"- {rs_id}")
    try:
        rs_obj = dataset.get_record_set(rs_id)
        fields = rs_obj['fields'] if 'fields' in rs_obj else []
        print("  Fields (@id):")
        for f in fields:
            print(f"    - {f['@id']} (name: {f.get('name', '')})")
    except Exception as e:
        print(f"  Could not describe fields for: {rs_id}")

# Show sample records from each record set
for rs_id in record_sets:
    print(f"\nSample from {rs_id}:")
    try:
        for i, row in enumerate(dataset.records(record_set=rs_id)):
            print(row)
            if i >= 1:
                break
    except Exception as e:
        print(f"  Error loading records: {e}")

## 3. Data Extraction
Load each record set into a DataFrame for further exploration. Use strict referencing via the exact `@id` for each record set.

_Note: If you want to work with a specific record set or field, copy its `@id` from the previous overview._

In [ ]:
# Build a dictionary of DataFrames for each available record set
dataframes = {}

for rs_id in record_sets:
    try:
        recs = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(recs)
        print(f"Loaded {len(recs)} records from record set {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Pick the primary protein record set for demonstration. Replace this @id with the one of interest.
example_rs_id = record_sets[0] if record_sets else None
if example_rs_id:
    print("\nColumns available in example record set (by field @id):")
    print(list(dataframes[example_rs_id].columns))
    print("\nSample data:")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple EDA on a numeric field. You can change `numeric_field_id` and `group_field_id` to those relevant to your data (referenced by their `@id`).

In [ ]:
# Example: Use the first numeric column found
example_df = dataframes[example_rs_id]
# Find a likely numeric field by column name
numeric_field_id = None
for col in example_df.columns:
    if any(s in col.lower() for s in ['intensity', 'abundance', 'count', 'coverage', 'peptide', 'mw', 'mass', 'weight']):
        if pd.api.types.is_numeric_dtype(example_df[col]):
            numeric_field_id = col
            break
# If not found, pick any numeric field
if not numeric_field_id:
    for col in example_df.columns:
        if pd.api.types.is_numeric_dtype(example_df[col]):
            numeric_field_id = col
            break

print(f"Numeric field chosen for EDA: {numeric_field_id}")
if numeric_field_id:
    numeric_series = pd.to_numeric(example_df[numeric_field_id], errors='coerce')
    threshold = np.nanmedian(numeric_series)
    filtered_df = example_df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > median ({threshold}): {len(filtered_df)} records.")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        numeric_series.loc[filtered_df.index] - numeric_series.mean()
    ) / numeric_series.std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group field to demonstrate grouping; e.g., 'description' or a categorical column
    group_field_id = None
    for col in example_df.columns:
        if col != numeric_field_id and example_df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Visualize a numeric field distribution and relationship to a group field if available.

In [ ]:
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    plt.hist(pd.to_numeric(example_df[numeric_field_id], errors='coerce').dropna(), bins=30, color='dodgerblue', alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        group_stats = example_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        group_stats.plot(kind='bar', figsize=(10,4), color='darkorchid', title=f'Top 10 {group_field_id} by mean {numeric_field_id}')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
- Successfully loaded and explored the FAIR² mass spectrometry protein dataset using `mlcroissant`, programmatically referencing fields and record sets by `@id` as recommended for reproducible FAIR data workflows.
- Inspected available record sets and columns, loaded data into DataFrames, filtered and normalized numeric fields, and visualized key distributions.
- For deeper biological questions, cross-reference protein accession and annotation fields to external resources.

**Remember:** All references in this notebook use exact `@id` fields, ensuring unambiguous and reproducible access to dataset elements.
